# **Setup**

In [1]:
# Estou utilizando o drive para carregar os datasets direto de lá
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install torch
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install torch-geometric
!pip install torch-geometric-temporal
!pip install numpy pandas tqdm scikit-learn matplotlib loguru torchmetrics timesfm

Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html


In [3]:
import sys
sys.path.append('/content/drive/MyDrive/ColabData/LABIA/DatasetsTSFM')

In [4]:
# basic
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
# pre processing
from sklearn import preprocessing as pre
# NN
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import torch.optim as optim
from torch.nn import MSELoss
from torch_geometric.nn import GCNConv
# val and plot
from torchmetrics.regression import R2Score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from loguru import logger as log
from val import calculate_metrics
# plot
import matplotlib.pyplot as plt
# foundation model
import timesfm
from functools import reduce



 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.11.12 (main, Apr  9 2025, 08:55:54) [GCC 11.4.0].


# **Experimento**

In [5]:
SEED = 1345
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
warnings.filterwarnings('ignore')

In [6]:
print(torch.__version__)
print(torch.version.cuda)

2.6.0+cu124
12.4


In [7]:
def load_datasets(filepath):
    """Carrega os datasets de arquivos pickle."""
    try:
        with open(filepath, 'rb') as f:
            dataset = pickle.load(f)
        return dataset
    except IOError as e:
        log.error(f"Erro ao carregar o dataset: {e}")
    except pickle.PickleError as e:
        log.error(f"Erro ao desserializar o dataset: {e}")
        traceback.print_exception(e)

In [8]:
test_dataset = load_datasets(f'/content/drive/MyDrive/ColabData/LABIA/DatasetsTSFM/test2.pkl')

In [9]:
# x (dados de input)
test_dataset[0].x.shape

torch.Size([2871, 280])

In [10]:
inference_size = test_dataset[0].y.shape[1]
inference_size

200

In [11]:
# 44783654 ufba ondina portaria 01
# 43768720 estacao da lapa
# 230565994 farol de itapua
# 125960550 estadio barradão
# 45833547 Ferry
# 44784438 fonte nova
# 47568123 shopping barra
# 44072192 tatro castro alves
# 258781031 rodovaria
# 44783914 elevador lacerda

In [12]:
ids = {53:'125960550', 365:'230565994', 382:'258781031', 666:'43768720', 701:'44072192', 1326:'44783654', 1404:'44783914', 1569:'44784438', 1916:'45833547', 2617:'47568123'}
nodes =  [53, 365, 382, 666, 701, 1326, 1404, 1569, 1916, 2617]

In [18]:
model_name = 'timesfm-2.0-500m-pytorch'

if model_name == 'timesfm-2.0-500m-pytorch':
  tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="gpu",
          per_core_batch_size=40,
          horizon_len=inference_size,
          num_layers=50,
          use_positional_embedding=False,
          context_len=2048,

      ),
      checkpoint=timesfm.TimesFmCheckpoint(
                    huggingface_repo_id=(''.join(('google/', model_name))),
      )
)

elif model_name == 'timesfm-1.0-200m-pytorch':
  tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="gpu",
          per_core_batch_size=40,
          horizon_len=inference_size,
      ),
  checkpoint=timesfm.TimesFmCheckpoint(
                    huggingface_repo_id=(''.join(('google/', model_name))),
      )
  )

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [19]:
scores_error = {'node':[], 'mae': [], 'mse': [], 'r2': [], 'mape': []}
targets = {}

for node in nodes:
    temp_scores_error = {'node':[], 'mae': [], 'mse': [], 'r2': [], 'mape': []}
    targets[node] = []
    cost, time = 0, 0
    for time, snapshot in tqdm(enumerate(test_dataset)):
        snapshot.to('cpu')

        input_data = np.array(snapshot.x[node, :])
        forecast, experimental_quantile_forecast = tfm.forecast(
            [input_data],
            freq=[0],
        )
        #
        y_hat =  torch.tensor(forecast[0])
        #

        # nao alterar mais abaixo

        cost = cost + torch.mean((y_hat-snapshot.y)**2)
        y_true = snapshot.y.cpu().data.numpy()
        y_pred = y_hat.cpu().data.numpy()

        temp_scores_error['node'].append(node)
        temp_scores_error['mse'].append(mean_squared_error(y_true[node,:], y_pred))
        temp_scores_error['mae'].append(mean_absolute_error(y_true[node,:], y_pred))
        temp_scores_error['r2'].append(r2_score(y_true[node,:], y_pred))
        temp_scores_error['mape'].append(mean_absolute_percentage_error(y_true[node,:], y_pred))

        #targets.append({'true': y_true,
                        #'pred': y_pred})

        targets[node].append({"input":  snapshot.x[node,:].cpu().numpy(),
                              'true': y_true,
                              'pred': y_pred,
                              'node': ids[node]
                             })

    # Calculate and print the averages of each metric
    for metric, values in temp_scores_error.items():
      average = sum(values) / len(values) if values else 0
      scores_error[metric].append(average)
      print(f"Average {metric.upper()}: {average:.4f}")



    cost = cost / (time+1)
    cost = cost.item()
    log.info(f"node: {node} MSE test: {cost:.4f}")


38it [00:26,  1.46it/s]
2025-04-29 14:12:57.162 | INFO     | __main__:<cell line: 0>:51 - node: 53 MSE test: 41918.2266


Average NODE: 53.0000
Average MAE: 3.9212
Average MSE: 28.6096
Average R2: -0.0540
Average MAPE: 0.3835


38it [00:26,  1.43it/s]
2025-04-29 14:13:23.771 | INFO     | __main__:<cell line: 0>:51 - node: 365 MSE test: 40540.0000


Average NODE: 365.0000
Average MAE: 11.1615
Average MSE: 262.6104
Average R2: -0.0432
Average MAPE: 0.8072


38it [00:27,  1.40it/s]
2025-04-29 14:13:50.878 | INFO     | __main__:<cell line: 0>:51 - node: 382 MSE test: 939092.1875


...

Average NODE: 1569.0000
Average MAE: 97.1807
Average MSE: 15431.7707
Average R2: -0.0692
Average MAPE: 0.7041


38it [00:28,  1.34it/s]
2025-04-29 14:16:38.625 | INFO     | __main__:<cell line: 0>:51 - node: 1916 MSE test: 35072.8828


Average NODE: 1916.0000
Average MAE: 41.0514
Average MSE: 2700.7949
Average R2: -0.2289
Average MAPE: 0.9100


38it [00:28,  1.33it/s]
2025-04-29 14:17:07.152 | INFO     | __main__:<cell line: 0>:51 - node: 2617 MSE test: 32773.0312


Average NODE: 2617.0000
Average MAE: 43.8454
Average MSE: 3119.4813
Average R2: 0.0470
Average MAPE: 0.4877


In [20]:
with open(f'{model_name}-targets-batch.pkl', 'wb') as f:
    pickle.dump(targets, f)

In [ ]:
df_results = pd.DataFrame(scores_error)
df_results

In [ ]:
df_results["model"] = "timesfm-1.0-200m"

In [ ]:
df_results[['model', 'node', 'mae', 'mse', 'r2', 'mape']]

In [ ]:
df_results.to_csv(''.join((model_name, '-batch.csv')))